# Entraînement du détecteur joueurs / gardiens / arbitres / ballon

Produit `player_detection.pt`, second des deux modèles nécessaires au pipeline.
L'autre est `pitch_keypoints.pt` (voir `train_keypoints_colab.ipynb`).

**Durée** : 2 h à 3 h sur GPU T4 gratuit. L'entrée est en 1280 px, pas 640 :
le ballon fait une douzaine de pixels et disparaît en dessous.

**Avant de lancer**, activer le GPU :
- Colab : *Exécution → Modifier le type d'exécution → GPU*
- Kaggle : panneau de droite, *Accelerator → GPU T4 x2*

Kaggle offre 30 h de GPU par semaine, contre un quota variable chez
Colab. Les deux fonctionnent ; ce notebook s'adapte seul.


## 1. GPU et installation


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout or
      "AUCUN GPU : Exécution -> Modifier le type d'exécution -> GPU")


In [ ]:
%pip install -q ultralytics roboflow pyyaml


## Dossier de travail

Colab et Kaggle n'écrivent pas au même endroit. Cette cellule choisit
le bon : sur Kaggle, écrire ailleurs que dans `/kaggle/working` remplit
un système de fichiers restreint et fait arrêter la session.


In [ ]:
from pathlib import Path

# Dossier de travail selon la plateforme. Sur Kaggle, tout ce qui est écrit
# hors de /kaggle/working atterrit sur un système de fichiers restreint :
# le dataset et les points de contrôle d'entraînement le remplissent, et la
# session est arrêtée en cours de route.
RACINE = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
RACINE.mkdir(parents=True, exist_ok=True)
print('dossier de travail :', RACINE)


## 2. Clé Roboflow


In [ ]:
import os
from getpass import getpass

# En mode différé (Save Version), getpass n'a aucune entrée clavier et le
# notebook resterait bloqué là sans fin. Les secrets Kaggle passent devant.
cle = os.environ.get('ROBOFLOW_API_KEY')
if not cle:
    try:
        from kaggle_secrets import UserSecretsClient   # Add-ons -> Secrets

        cle = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    except Exception:
        cle = getpass('Clé Roboflow : ')
os.environ['ROBOFLOW_API_KEY'] = cle


## 3. Télécharger

`football-players-detection-3zvbc` v20. Bien plus fourni que la v1 utilisée
au début du projet.


In [ ]:
from roboflow import Roboflow
from pathlib import Path

rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
projet = rf.workspace('roboflow-jvuqo').project('football-players-detection-3zvbc')
dataset = projet.version(20).download('yolov8', location=str(RACINE / 'players_ds'), overwrite=False)
print('téléchargé dans', dataset.location)
print('instances par classe :', projet.classes)


## 4. Garde-fou : l'ordre des classes

`config.py` fige les identifiants : ballon 0, gardien 1, joueur 2, arbitre 3.
Si le dataset les ordonne autrement, le pipeline confondra joueurs et arbitres
sans qu'aucune erreur ne le signale — les arbitres seraient comptés comme des
joueurs, et les gardiens ne seraient jamais rattachés à une équipe.

En cas d'échec ici, corrige les constantes de `config.py` plutôt que le
dataset : c'est le code qui doit suivre les poids.


In [ ]:
import yaml

ATTENDU = ['ball', 'goalkeeper', 'player', 'referee']   # cf. config.py

yaml_path = Path(dataset.location) / 'data.yaml'
cfg = yaml.safe_load(yaml_path.read_text())
noms = cfg['names']
if isinstance(noms, dict):
    noms = [noms[i] for i in sorted(noms)]

assert noms == ATTENDU, (
    'ORDRE DES CLASSES DIVERGENT : ne pas entraîner tel quel.\n'
    f'dataset : {noms}\nattendu : {ATTENDU}\n'
    'Corriger BALL_ID / GOALKEEPER_ID / PLAYER_ID / REFEREE_ID dans config.py.')
print('ordre des classes conforme :', noms)


## 5. Corriger les chemins

Même défaut que pour le dataset de points clés : Roboflow écrit des chemins
relatifs qu'Ultralytics résout un cran trop haut.


In [ ]:
cfg['path'] = str(Path(dataset.location).resolve())
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'
yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

for split in ('train', 'valid', 'test'):
    n = len(list((Path(cfg['path']) / split / 'images').glob('*')))
    print(f'{split:6s}: {n} images')


## 6. Entraîner

**`imgsz=1280` est le réglage décisif.** Sur une vue de diffusion, le ballon
mesure 12 à 19 pixels. En 640, il disparaît, et la possession avec lui.

Le déséquilibre des classes est extrême : environ 400 ballons annotés contre
12 000 joueurs, soit 30 pour 1. Le modèle sera naturellement bien meilleur sur
les joueurs. C'est pourquoi la cellule 8 mesure le ballon séparément — une
mAP globale flatteuse peut masquer un ballon jamais détecté.

La mosaïque est ici **utile**, contrairement au modèle de points clés : elle
varie les contextes de joueurs. `close_mosaic=10` la coupe pour les dix
dernières époques, afin de finir sur des images réalistes.

`batch=8` tient dans les 15 Go d'un T4 en 1280 px. En cas de mémoire
insuffisante, descendre à 4.


In [ ]:
from ultralytics import YOLO

MODELE = 'yolov8m.pt'    # 'yolov8l.pt' si le T4 le supporte

resultats = YOLO(MODELE).train(
    data=str(yaml_path),
    epochs=100,
    patience=25,
    imgsz=1280,        # non négociable : voir ci-dessus
    batch=8,
    project=str(RACINE / 'runs'),
    name='players',
    mosaic=1.0,
    close_mosaic=10,
    plots=True,
)


## 7. Métriques par classe

La ligne qui compte est celle du ballon, pas la moyenne.


In [ ]:
best = RACINE / 'runs' / 'players' / 'weights' / 'best.pt'
print('poids :', best, '|', round(best.stat().st_size / 1e6, 1), 'Mo')

modele = YOLO(str(best))
m = modele.val(data=str(yaml_path), split='val', imgsz=1280)
print()
for i, nom in enumerate(ATTENDU):
    p, r, ap50, ap = m.box.class_result(i)
    print(f'{nom:11s} précision {p:.3f}  rappel {r:.3f}  mAP50 {ap50:.3f}')
print(f'\nmAP50 global : {m.box.map50:.3f}')


## 8. Test d'acceptation : le ballon

La possession dépend entièrement de la détection du ballon. Repère mesuré
avec un YOLO générique COCO, non spécialisé : 1 image sur 5 en 720p,
7 sur 10 en 1080p. Ce modèle doit faire nettement mieux.

Un rappel ballon sous 0,5 rend la possession peu fiable — le rapport devra
le signaler au club plutôt que d'afficher un chiffre inventé.


In [ ]:
import numpy as np

BALLON = ATTENDU.index('ball')
images = sorted((Path(cfg['path'])/'valid'/'images').glob('*'))
labels_dir = Path(cfg['path'])/'valid'/'labels'

avec_ballon = trouves = 0
tailles = []
for img in images:
    lab = labels_dir / (img.stem + '.txt')
    if not lab.exists():
        continue
    verite = [l.split() for l in lab.read_text().splitlines() if l.strip()]
    if not any(int(l[0]) == BALLON for l in verite):
        continue
    avec_ballon += 1

    r = modele.predict(str(img), imgsz=1280, conf=0.25, verbose=False)[0]
    classes = r.boxes.cls.cpu().numpy().astype(int)
    boites = r.boxes.xyxy.cpu().numpy()[classes == BALLON]
    if len(boites):
        trouves += 1
        tailles.append(float(np.mean(boites[:,2]-boites[:,0])))

rappel = trouves / avec_ballon if avec_ballon else 0.0
print(f'images contenant un ballon : {avec_ballon}')
print(f'ballon retrouvé            : {trouves}  ({rappel:.0%})')
if tailles:
    print(f'largeur médiane du ballon  : {np.median(tailles):.0f} px')
print()
if rappel >= 0.8:
    print('BON — possession exploitable.')
elif rappel >= 0.5:
    print('PASSABLE — possession indicative, à signaler comme telle au club.')
else:
    print('INSUFFISANT — ne pas vendre de possession. Réessayer en yolov8l, ou imgsz=1536.')


## Test sur tes propres images *(facultatif)*

Cette cellule ne sert pas à l'entraînement : elle mesure ce que le modèle
donne sur **tes** vidéos. Tu peux la sauter sans conséquence.

Exécute-la, clique sur **Choisir les fichiers**, et sélectionne l'archive
`images_test_domaine.zip`. Elle est décompressée automatiquement.


In [ ]:
import shutil, zipfile
from pathlib import Path

dossier = RACINE / 'mes_images'
dossier.mkdir(parents=True, exist_ok=True)

try:
    # Colab : sélecteur de fichiers du navigateur.
    from google.colab import files as _colab

    for nom in _colab.upload():
        chemin = Path(nom)
        if chemin.suffix.lower() == '.zip':
            with zipfile.ZipFile(chemin) as archive:
                archive.extractall(dossier)
            chemin.unlink()
        else:
            shutil.move(nom, dossier / nom)
except ImportError:
    # Kaggle : pas de sélecteur. Ajouter l'archive comme Dataset dans le
    # panneau de droite, puis exécuter — elle apparaît sous /kaggle/input.
    for source in Path('/kaggle/input').rglob('*'):
        if source.suffix.lower() == '.zip':
            with zipfile.ZipFile(source) as archive:
                archive.extractall(dossier)
        elif source.suffix.lower() in ('.png', '.jpg', '.jpeg'):
            shutil.copy(source, dossier / source.name)

IMAGES = ('.png', '.jpg', '.jpeg')
fichiers = sorted(p for p in dossier.rglob('*') if p.suffix.lower() in IMAGES)
print(f'{len(fichiers)} image(s) trouvée(s)')

import cv2
avec = 0
for f in fichiers:
    r = modele.predict(str(f), imgsz=1280, conf=0.25, verbose=False)[0]
    c = r.boxes.cls.cpu().numpy().astype(int)
    compte = {nom: int((c == i).sum()) for i, nom in enumerate(ATTENDU)}
    avec += compte['ball'] > 0
    print(f"{f.name:44s} joueurs {compte['player']:2d}  gardiens {compte['goalkeeper']}  "
          f"arbitres {compte['referee']}  ballon {compte['ball']}")

if fichiers:
    print(f'\\nballon trouvé sur {avec}/{len(fichiers)} images = {100*avec/len(fichiers):.0f}%')


## Récupérer les poids

- **Colab** : le téléchargement démarre tout seul.
- **Kaggle** : la cellule affiche un lien de téléchargement direct. Il fonctionne
  dans la session en cours, sans rien relancer.

Ne pas passer par *Save Version → Save & Run All* pour récupérer un fichier :
ce bouton **réexécute le notebook entier**, soit plusieurs heures
d'entraînement refaites pour rien. Si un lien direct échoue, utiliser
*Quick Save*, qui enregistre la session telle quelle, puis l'onglet *Output*
de la version ainsi créée.

Le fichier porte déjà le nom attendu par `config.py` : le déposer tel quel
dans `models/` du dépôt.


In [ ]:
from pathlib import Path
import shutil

NOM = 'player_detection.pt'    # nom attendu par football_analysis/config.py

cible = RACINE / NOM
shutil.copy(best, cible)
print(f'{cible} — {cible.stat().st_size / 1e6:.1f} Mo')

try:
    from google.colab import files as _colab

    _colab.download(str(cible))
except ImportError:
    # Kaggle : rien ne peut déclencher un téléchargement depuis Python, et
    # l'onglet Output d'une session interactive est vide — il n'est rempli
    # que par une version enregistrée. Ce lien-ci sert le fichier directement.
    from IPython.display import FileLink, display

    display(FileLink(str(cible.relative_to(RACINE))))
